# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesador para árboles sin escalado
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), feature_columns),
])

# Preprocesador con imputación y escalado para modelos sensibles a la escala
scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns),
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

Porque si el escalador se ajusta con todo el entrenamiento antes de la validación cruzada, la media y la desviación de cada fold de validación ya influyen en la transformación. Eso filtra información de validación hacia el entrenamiento y produce métricas optimistas. Dentro del pipeline el escalador se ajusta solo con los folds de entrenamiento en cada iteración, igual que ocurriría en producción.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("prep", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("prep", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("prep", scaled_preprocessor),
        ("model", KNeighborsClassifier()),
    ]),
}

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(clone(pipeline), X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False)
initial_comparison

,model,train_accuracy,val_accuracy,val_std
1,logistic_regression,1.0000,0.979064,0.027713
2,knn,0.9771,0.950739,0.035274
0,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1, 10],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(pipeline, param_grids[name], cv=cv, scoring="accuracy", return_train_score=True)
    search.fit(X_train, y_train)
    searches[name] = search
    best = search.cv_results_["rank_test_score"].argmin()
    rows.append({
        "model": name,
        "best_params": search.best_params_,
        "cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
    })

tuned_comparison = pd.DataFrame(rows).sort_values("cv_accuracy", ascending=False)
tuned_comparison

,model,best_params,cv_accuracy,cv_std,train_accuracy
1,logistic_regression,{'model__C': 1},0.979064,0.027713,1.000000
2,knn,"{'model__n_neighbors': 7, 'model__weights': 'u...",0.957882,0.040520,0.964773
0,decision_tree,"{'model__max_depth': 5, 'model__min_samples_le...",0.902463,0.076225,0.985934


## 6. Selección y persistencia del campeón

In [8]:
# En caso de empate se prefiere el modelo con menor desviación estándar
champion_name = tuned_comparison.sort_values(["cv_accuracy", "cv_std"], ascending=[False, True]).iloc[0]["model"]
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name, searches[champion_name].best_params_)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_folds": 5,
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "n_train": int(len(X_train)),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

cv_frames = []
for name, search in searches.items():
    frame = pd.DataFrame(search.cv_results_)
    frame.insert(0, "model", name)
    cv_frames.append(frame[["model", "params", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]])
pd.concat(cv_frames).to_csv(REPORTS_DIR / "cv_results.csv", index=False)

Campeón: logistic_regression {'model__C': 1}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

Se seleccionó la regresión logística con C igual a 1. Obtuvo la mayor accuracy de validación cruzada, cerca de 0.979, y también la menor desviación entre folds, lo que indica un desempeño estable. Su accuracy de entrenamiento es 1.0, pero la brecha con validación es pequeña, mientras que el árbol de decisión muestra mayor sobreajuste y variabilidad. Además es un modelo simple, rápido y más interpretable que KNN, ya que sus coeficientes indican qué variables empujan cada clase.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.